In [1]:
import cv2
import mediapipe as mp
import numpy as np
from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import matplotlib as mpl
import os
from matplotlib.animation import PillowWriter

mpl.rcParams['animation.embed_limit'] = 50 

import matplotlib.animation as animation
animation.writers.list()

['pillow', 'ffmpeg', 'ffmpeg_file', 'html']

In [2]:
def create_overlay_animation(video_path, model_path):

    base_options = python.BaseOptions(model_asset_path=model_path)

    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    JOINTS = [
        0,   # nose/head
        11, 12,
        13, 14,
        15, 16,
        23, 24,
        25, 26,
        27, 28
    ]

    CONNECTIONS = [
        (0,1), (0,2),
        (1,3), (3,5),
        (2,4), (4,6),
        (1,2),
        (1,7), (2,8),
        (7,8),
        (7,9), (9,11),
        (8,10), (10,12)
    ]

    frames = []

    with vision.PoseLandmarker.create_from_options(options) as landmarker:

        cap = cv2.VideoCapture(video_path)

        fps = cap.get(cv2.CAP_PROP_FPS)

        frame_idx = 0

        while cap.isOpened():

            ret, frame = cap.read()

            if not ret:
                break

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=image_rgb
            )

            timestamp_ms = int((frame_idx / fps) * 1000)

            results = landmarker.detect_for_video(
                mp_image,
                timestamp_ms
            )

            output_frame = image_rgb.copy()

            if results.pose_landmarks:

                landmarks = results.pose_landmarks[0]

                points = []

                h, w, _ = output_frame.shape

                for idx in JOINTS:

                    lm = landmarks[idx]

                    x = int(lm.x * w)
                    y = int(lm.y * h)

                    points.append((x, y))

                    cv2.circle(
                        output_frame,
                        (x, y),
                        5,
                        (255, 0, 0),
                        -1
                    )

                for start, end in CONNECTIONS:

                    cv2.line(
                        output_frame,
                        points[start],
                        points[end],
                        (0,255,0),
                        2
                    )

            frames.append(output_frame)

            frame_idx += 1

        cap.release()

    fig = plt.figure(figsize=(8,6))

    img_plot = plt.imshow(frames[0])

    plt.axis("off")

    def update(i):

        img_plot.set_data(frames[i])

        return [img_plot]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=50,
        blit=True
    )

    plt.close(fig)

    return anim

In [3]:
# paths
video_dir = "../../MainProject/Assignment14"
model_path = "../../EnisProject/data/pose_landmarker.task"

# output folder
output_dir = "./overlay_outputs"
os.makedirs(output_dir, exist_ok=True)

# explicitly create A1.avi -> A44.avi
videos = [f"A{i}.avi" for i in range(1, 45)]

print(f"Processing {len(videos)} videos...")

for i, vid in enumerate(videos):

    video_path = os.path.join(video_dir, vid)

    # skip missing files safely
    if not os.path.exists(video_path):
        print(f"Skipping missing file: {vid}")
        continue

    print(f"\n[{i+1}/{len(videos)}] Processing: {vid}")

    # create animation
    anim = create_overlay_animation(
        video_path,
        model_path
    )

    # output path
    output_path = os.path.join(
        output_dir,
        vid.replace(".avi", ".gif")
    )

    # save without ffmpeg
    writer = PillowWriter(fps=20)

    anim.save(output_path, writer=writer)

    print(f"Saved: {output_path}")

    # free memory
    plt.close('all')

print("\nFinished processing all videos.")


#anim = create_overlay_animation(
#    video_path,
#    model_path
#)

#HTML(anim.to_html5_video())
#HTML(anim.to_jshtml())

Processing 44 videos...
Skipping missing file: A1.avi
Skipping missing file: A2.avi
Skipping missing file: A3.avi
Skipping missing file: A4.avi
Skipping missing file: A5.avi
Skipping missing file: A6.avi
Skipping missing file: A7.avi
Skipping missing file: A8.avi
Skipping missing file: A9.avi
Skipping missing file: A10.avi
Skipping missing file: A11.avi
Skipping missing file: A12.avi
Skipping missing file: A13.avi
Skipping missing file: A14.avi
Skipping missing file: A15.avi
Skipping missing file: A16.avi
Skipping missing file: A17.avi
Skipping missing file: A18.avi
Skipping missing file: A19.avi
Skipping missing file: A20.avi
Skipping missing file: A21.avi
Skipping missing file: A22.avi
Skipping missing file: A23.avi
Skipping missing file: A24.avi
Skipping missing file: A25.avi
Skipping missing file: A26.avi
Skipping missing file: A27.avi
Skipping missing file: A28.avi
Skipping missing file: A29.avi
Skipping missing file: A30.avi
Skipping missing file: A31.avi
Skipping missing file: A